In [1]:
from astropy.io import fits
import pandas as pd

def analyze_fits(file_path):
    print(f"--- 正在分析文件: {file_path} ---\n")
    
    with fits.open(file_path) as hdul:
        # 1. 打印 FITS 文件的总体结构 (HDU 信息)
        print("【1. 文件结构 (HDU List)】")
        hdul.info()
        print("\n")

        for i, hdu in enumerate(hdul):
            print(f"--- 扩展层 (Extension) {i} 详情 ---")
            header = hdu.header
            
            # 2. 识别数据类型 (图像 or 表格)
            if isinstance(hdu, (fits.ImageHDU, fits.PrimaryHDU)) and hdu.data is not None:
                print(f"类型: 图像 (Image)")
                print(f"维度 (Shape): {hdu.data.shape}")
                print(f"数据类型 (Dtype): {hdu.data.dtype}")
                # 打印关键天文坐标信息 (WCS)
                if 'CRVAL1' in header:
                    print(f"参考坐标 (RA, Dec): ({header['CRVAL1']}, {header['CRVAL2']})")
                if 'PHOTZP' in header or 'MAGZERO' in header:
                    zp = header.get('PHOTZP') or header.get('MAGZERO')
                    print(f"测光零点 (Photometric Zeropoint): {zp}")

            elif isinstance(hdu, (fits.BinTableHDU, fits.TableHDU)):
                print(f"类型: 二进制表格 (Binary Table/Catalog)")
                print(f"行数 (Rows): {len(hdu.data)}")
                print(f"列数 (Columns): {len(hdu.columns)}")
                
                # 展示列名，方便确认包含哪些物理量
                col_names = hdu.columns.names
                print(f"列名: {col_names}")
            
            # 如果想看完整的 Header，可以取消下面这行的注释
            # print("\n")
            # print(header)
            print("-" * 40 + "\n")

In [8]:
# 使用
analyze_fits("../data_Clauds/COSMOS-HSCpipe-Phosphoros.fits")

--- 正在分析文件: ../data_Clauds/COSMOS-HSCpipe-Phosphoros.fits ---

【1. 文件结构 (HDU List)】
Filename: ../data_Clauds/COSMOS-HSCpipe-Phosphoros.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  /Users/desprez/CLAUDS/DR2/combCats/COSMOS-HSCpipe.fits#1    1 BinTableHDU    483   5474883R x 235C   [K, D, D, 5A, 3A, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, L, 7L, I, L, L, L, L, L, D, L

In [9]:
analyze_fits("../data_Clauds/XMMLSS-HSCpipe-Phosphoros.fits")

--- 正在分析文件: ../data_Clauds/XMMLSS-HSCpipe-Phosphoros.fits ---

【1. 文件结构 (HDU List)】
Filename: ../data_Clauds/XMMLSS-HSCpipe-Phosphoros.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  /Volumes/Magmar/CLAUDS-HSC/catalogs/DR2_CLAUDS/XMM-LSS-HSCpipe.fi...    1 BinTableHDU    483   5310123R x 235C   [K, D, D, 4A, 3A, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, E, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, D, D, D, D, D, D, D, D, D, D, D, L, L, L, L, L, L, L, L, 7L, I, L, L, L

In [ ]:

import numpy as np
from astropy.io import fits
from astropy.table import Table
from pathlib import Path
from typing import Union, Tuple

# ============================================================================
# 波段定义
# ============================================================================

FILTERS = ['MegaCam-u', 'MegaCam-uS', 'HSC-G', 'HSC-R', 'HSC-I', 'HSC-Z', 'HSC-Y']

# 测光列对
PHOTOMETRY_PAIRS = [
    ('FLUX_CMODEL', 'FLUXERR_CMODEL'),
]

# 清洁标志列
CLEAN_FLAGS = [
    'isClean', 'hasBadPhotometry', 'isDuplicated',
    'isParent', 'isNoData', 'isSky', 'notObserved'
]

# 全局列
GLOBAL_COLS = [
    'ID', 'RA', 'DEC', 'tract', 'patch',
    'isStar', 'isStarTemp', 'isCompact',
    'isOutsideMask', 'FLAG_FIELD_BINARY',
    'ZPHOT', 'Z_LOW68', 'Z_HIGH68', 'Z_CHI'
]


# ============================================================================
# 数据读取函数
# ============================================================================

def read_fits_data(fits_path: Union[str, Path], hdu: int = 1) -> Table:
    """
    从FITS文件读取CLAUDS+HSC数据，处理多维数组列。
    
    Args:
        fits_path: FITS文件路径
        hdu: FITS HDU索引（默认1）
    
    Returns:
        astropy.table.Table: 提取的数据表
    """
    fits_path = Path(fits_path)
    if not fits_path.exists():
        raise FileNotFoundError(f"文件不存在: {fits_path}")
    
    with fits.open(fits_path) as hdul:
        data = hdul[hdu].data
        available_cols = set(data.dtype.names)
        
        output = Table()
        
        # ===== 1. 提取全局列 =====
        for col in GLOBAL_COLS:
            if col not in available_cols:
                continue
            col_data = data[col]
            # 检查维度：若为多维则取第一列
            if col_data.ndim > 1:
                col_data = col_data[:, 0]
            output[col] = col_data
        
        # ===== 2. 提取测光数据（波段循环） =====
        for filt in FILTERS:
            for flux_prefix, err_prefix in PHOTOMETRY_PAIRS:
                flux_col = f'{flux_prefix}_{filt}'
                err_col = f'{err_prefix}_{filt}'
                
                # 提取通量
                if flux_col in available_cols:
                    flux_data = data[flux_col]
                    if flux_data.ndim > 1:
                        flux_data = flux_data[:, 0]
                    output[flux_col] = flux_data
                
                # 提取误差
                if err_col in available_cols:
                    err_data = data[err_col]
                    if err_data.ndim > 1:
                        err_data = err_data[:, 0]
                    output[err_col] = err_data
        
        # ===== 3. 提取清洁标志 =====
        for filt in FILTERS:
            for flag in CLEAN_FLAGS:
                flag_col = f'{flag}_{filt}'
                if flag_col in available_cols:
                    flag_data = data[flag_col]
                    if flag_data.ndim > 1:
                        flag_data = flag_data[:, 0]
                    output[flag_col] = flag_data
    
    print(f"✓ 成功读取: {len(output):,} 行, {len(output.colnames)} 列")
    return output


def filter_quality_strict(table: Table) -> Table:
    """
    严格的质量过滤掩模。
    
    过滤条件:
    - 所有有数据的波段 isClean == True（确保所有探测波段数据质量高）
    - isStar == False（非恒星）
    - 所有有数据的波段 isParent == False（避免父天体重复计数）
    - isOutsideMask == True（在有效观测范围内）
    
    Args:
        table: 输入表
    
    Returns:
        Table: 过滤后的表
    """
    n_before = len(table)
    mask = np.ones(len(table), dtype=bool)
    
    # 1. 恒星剔除（全局）
    if 'isStar' in table.colnames:
        mask &= (table['isStar'] == False)
    
    # 2. 掩模范围（全局，确保在有效观测范围内）
    if 'isOutsideMask' in table.colnames:
        mask &= (table['isOutsideMask'] == True)
    
    # 3. isClean 检查：所有波段都要求 == True
    filters = ['MegaCam-u', 'MegaCam-uS', 'HSC-G', 'HSC-R', 'HSC-I', 'HSC-Z', 'HSC-Y']
    for filt in filters:
        col = f'isClean_{filt}'
        if col in table.colnames:
            mask &= (table[col] == True)
            clean_count = np.sum(table[col])
            print(f"  {col}: {clean_count:,} / {n_before:,}")
    
    # 4. isParent 检查：所有波段都要求 == False（或只需i波段）
    for filt in filters:
        col = f'isParent_{filt}'
        if col in table.colnames:
            mask &= (table[col] == False)
    
    table_filtered = table[mask]
    n_after = len(table_filtered)
    
    print(f"\n严格质量过滤结果:")
    print(f"  过滤前: {n_before:,} 行")
    print(f"  过滤后: {n_after:,} 行")
    print(f"  保留率: {100*n_after/n_before:.1f}%")
    
    return table_filtered


# ============================================================================
# 测光处理函数
# ============================================================================

def flux_to_magnitude(flux: np.ndarray, 
                      flux_err: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    将通量转换为AB星等。
    
    公式: mag = -2.5 * log10(flux) + 31.4
    
    Args:
        flux: 通量 [nanoJansky]
        flux_err: 通量误差
    
    Returns:
        Tuple: (星等, 星等误差)
    """
    ZP_AB = 31.4
    
    # 避免log(0)
    safe_flux = np.clip(flux, 1e-10, None)
    mag = -2.5 * np.log10(safe_flux) + ZP_AB
    
    # 误差传播
    mag_err = np.abs(-2.5 / np.log(10) * flux_err / safe_flux)
    
    return mag, mag_err


def compute_color(table: Table, 
                 filter_short: str, 
                 filter_long: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    计算两波段间的颜色指数。
    
    Args:
        table: 数据表
        filter_short: 短波段 (e.g., 'MegaCam-u')
        filter_long: 长波段 (e.g., 'HSC-G')
    
    Returns:
        Tuple: (颜色, 颜色误差)
    """
    flux1_col = f'FLUX_CMODEL_{filter_short}'
    err1_col = f'FLUXERR_CMODEL_{filter_short}'
    flux2_col = f'FLUX_CMODEL_{filter_long}'
    err2_col = f'FLUXERR_CMODEL_{filter_long}'
    
    if not all(col in table.colnames for col in [flux1_col, err1_col, flux2_col, err2_col]):
        raise ValueError(f"缺少必要列: {filter_short} 或 {filter_long}")
    
    mag1, mag1_err = flux_to_magnitude(table[flux1_col].data, table[err1_col].data)
    mag2, mag2_err = flux_to_magnitude(table[flux2_col].data, table[err2_col].data)
    
    color = mag1 - mag2
    color_err = np.sqrt(mag1_err**2 + mag2_err**2)
    
    return color, color_err


# ============================================================================
# 统计与诊断函数
# ============================================================================

def print_statistics(table: Table) -> None:
    """打印数据表的统计信息。"""
    print(f"\n数据表统计:")
    print(f"  总行数: {len(table):,}")
    print(f"  总列数: {len(table.colnames)}")
    
    if 'isStar' in table.colnames:
        n_star = np.sum(table['isStar'] == True)
        n_galaxy = np.sum(table['isStar'] == False)
        print(f"\n天体类型统计:")
        print(f"  恒星 (isStar=True): {n_star:,}")
        print(f"  星系 (isStar=False): {n_galaxy:,}")
    
    if 'ZPHOT' in table.colnames:
        z_valid = table['ZPHOT'][table['ZPHOT'] > 0]
        if len(z_valid) > 0:
            print(f"\n红移统计:")
            print(f"  有效红移数: {len(z_valid):,}")
            print(f"  红移范围: {np.min(z_valid):.3f} - {np.max(z_valid):.3f}")
            print(f"  中位红移: {np.median(z_valid):.3f}")


def check_column_coverage(table: Table, filters: list = FILTERS) -> None:
    """检查各波段的列覆盖情况。"""
    print(f"\n列覆盖情况 (前5个波段):")
    for filt in filters[:5]:
        flux_col = f'FLUX_CMODEL_{filt}'
        clean_col = f'isClean_{filt}'
        
        flux_exist = '✓' if flux_col in table.colnames else '✗'
        clean_exist = '✓' if clean_col in table.colnames else '✗'
        
        print(f"  {filt:15s}: 测光{flux_exist} | 清洁{clean_exist}")


In [ ]:
'''
# ============================================================================
# 使用示例
# ============================================================================

if __name__ == "__main__":
    # 1. 读取FITS文件
    fits_file = "../data_Clauds/COSMOS-HSCpipe-Phosphoros.fits"
    table = read_fits_data(fits_file)
    
    # 2. 打印列信息
    print(f"\n提取的列名 ({len(table.colnames)} 列):")
    for i, col in enumerate(table.colnames, 1):
        print(f"{i:3d}. {col}")
    
    # 3. 检查列覆盖
    check_column_coverage(table)
    
    # 4. 统计信息
    print_statistics(table)
    
    # 5. 应用质量过滤
    table_clean = filter_quality(table)
    
    # 6. 计算颜色指数
    color_ug, color_ug_err = compute_color(table_clean, 'MegaCam-u', 'HSC-G')
    table_clean['color_u-g'] = color_ug
    table_clean['color_u-g_err'] = color_ug_err
    
    print(f"\n✓ 颜色计算完毕")
    print(f"  u-g 颜色范围: {np.min(color_ug):.2f} - {np.max(color_ug):.2f}")

'''

'\nif __name__ == "__main__":\n    # 单文件读取\n    fits_file = "/path/to/your/data.fits"\n    df = read_fits_selective(fits_file)\n    \n    # 应用质量过滤\n    df_clean = apply_quality_filter(df)\n    \n    # 计算颜色指数\n    color_ug, color_ug_err = compute_colors(df_clean, \'MegaCam-u\', \'HSC-G\')\n    df_clean[\'color_u-g\'] = color_ug\n    df_clean[\'color_u-g_err\'] = color_ug_err\n    \n    print(f"\n数据准备完毕: {len(df_clean):,} 个高质量候选者")\n    print(f"红移范围: {df_clean[\'ZPHOT\'].min():.2f} - {df_clean[\'ZPHOT\'].max():.2f}")\n'

In [16]:
fits_file = "../data_Clauds/COSMOS-HSCpipe-Phosphoros.fits"
table = read_fits_data(fits_file)

✓ 成功读取: 5,474,883 行, 77 列


In [20]:
print(f"\n提取的列名 ({len(table.colnames)} 列):")
for i, col in enumerate(table.colnames, 1):
    print(f"{i:3d}. {col}")

check_column_coverage(table)

print_statistics(table)


提取的列名 (77 列):
  1. ID
  2. RA
  3. DEC
  4. tract
  5. patch
  6. isStar
  7. isStarTemp
  8. isCompact
  9. isOutsideMask
 10. FLAG_FIELD_BINARY
 11. ZPHOT
 12. Z_LOW68
 13. Z_HIGH68
 14. Z_CHI
 15. FLUX_CMODEL_MegaCam-u
 16. FLUXERR_CMODEL_MegaCam-u
 17. FLUX_CMODEL_MegaCam-uS
 18. FLUXERR_CMODEL_MegaCam-uS
 19. FLUX_CMODEL_HSC-G
 20. FLUXERR_CMODEL_HSC-G
 21. FLUX_CMODEL_HSC-R
 22. FLUXERR_CMODEL_HSC-R
 23. FLUX_CMODEL_HSC-I
 24. FLUXERR_CMODEL_HSC-I
 25. FLUX_CMODEL_HSC-Z
 26. FLUXERR_CMODEL_HSC-Z
 27. FLUX_CMODEL_HSC-Y
 28. FLUXERR_CMODEL_HSC-Y
 29. isClean_MegaCam-u
 30. hasBadPhotometry_MegaCam-u
 31. isDuplicated_MegaCam-u
 32. isParent_MegaCam-u
 33. isNoData_MegaCam-u
 34. isSky_MegaCam-u
 35. notObserved_MegaCam-u
 36. isClean_MegaCam-uS
 37. hasBadPhotometry_MegaCam-uS
 38. isDuplicated_MegaCam-uS
 39. isParent_MegaCam-uS
 40. isNoData_MegaCam-uS
 41. isSky_MegaCam-uS
 42. notObserved_MegaCam-uS
 43. isClean_HSC-G
 44. hasBadPhotometry_HSC-G
 45. isDuplicated_HSC-G
 46. is

In [21]:
# 保存table为CSV文件
output_path = "../data_Clauds/COSMOS-HSCpipe-Phosphoros_extracted.csv"
table.write(output_path, format='ascii.csv', overwrite=True)
print(f"✓ 数据已保存到: {output_path}")
print(f"  行数: {len(table):,}")
print(f"  列数: {len(table.colnames)}")

✓ 数据已保存到: ../data_Clauds/COSMOS-HSCpipe-Phosphoros_extracted.csv
  行数: 5,474,883
  列数: 77


In [22]:
import pandas as pd

csv_file = "../data_Clauds/COSMOS-HSCpipe-Phosphoros_extracted.csv"
df = pd.read_csv(csv_file)

print(f"文件: {csv_file}")
print(f"总行数: {len(df):,} | 总列数: {len(df.columns)}")
print(f"\n列名 ({len(df.columns)} 列):\n{list(df.columns)}")
print(f"\n前5行:\n")
print(df.head())
print(f"\n数据类型:\n{df.dtypes}")

文件: ../data_Clauds/COSMOS-HSCpipe-Phosphoros_extracted.csv
总行数: 5,474,883 | 总列数: 77

列名 (77 列):
['ID', 'RA', 'DEC', 'tract', 'patch', 'isStar', 'isStarTemp', 'isCompact', 'isOutsideMask', 'FLAG_FIELD_BINARY', 'ZPHOT', 'Z_LOW68', 'Z_HIGH68', 'Z_CHI', 'FLUX_CMODEL_MegaCam-u', 'FLUXERR_CMODEL_MegaCam-u', 'FLUX_CMODEL_MegaCam-uS', 'FLUXERR_CMODEL_MegaCam-uS', 'FLUX_CMODEL_HSC-G', 'FLUXERR_CMODEL_HSC-G', 'FLUX_CMODEL_HSC-R', 'FLUXERR_CMODEL_HSC-R', 'FLUX_CMODEL_HSC-I', 'FLUXERR_CMODEL_HSC-I', 'FLUX_CMODEL_HSC-Z', 'FLUXERR_CMODEL_HSC-Z', 'FLUX_CMODEL_HSC-Y', 'FLUXERR_CMODEL_HSC-Y', 'isClean_MegaCam-u', 'hasBadPhotometry_MegaCam-u', 'isDuplicated_MegaCam-u', 'isParent_MegaCam-u', 'isNoData_MegaCam-u', 'isSky_MegaCam-u', 'notObserved_MegaCam-u', 'isClean_MegaCam-uS', 'hasBadPhotometry_MegaCam-uS', 'isDuplicated_MegaCam-uS', 'isParent_MegaCam-uS', 'isNoData_MegaCam-uS', 'isSky_MegaCam-uS', 'notObserved_MegaCam-uS', 'isClean_HSC-G', 'hasBadPhotometry_HSC-G', 'isDuplicated_HSC-G', 'isParent_HSC-G

In [2]:
analyze_fits("../CLAUDS_udrop_specz/COSMOS_udrop_specz.fits")

--- 正在分析文件: ../CLAUDS_udrop_specz/COSMOS_udrop_specz.fits ---

【1. 文件结构 (HDU List)】
Filename: ../CLAUDS_udrop_specz/COSMOS_udrop_specz.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  SPECINFO      1 BinTableHDU    286   4486R x 103C   [K, J, D, D, 22A, E, E, E, K, B, 3A, D, J, E, K, K, K, K, K, K, K, K, K, D, D, I, E, I, I, E, E, E, E, E, D, E, D, E, D, D, D, K, 108A, K, 10A, 209A, E, E, L, L, L, 30A, 100A, L, L, L, E, E, 10A, 450A, D, D, K, D, 10D, 4A, K, 6A, 20A, K, D, 9D, 54A, 9D, 120A, D, L, D, D, D, D, D, D, D, D, L, D, D, D, D, D, D, D, D, L, D, D, D, D, D, D, D, D]   


--- 扩展层 (Extension) 0 详情 ---
----------------------------------------

--- 扩展层 (Extension) 1 详情 ---
类型: 二进制表格 (Binary Table/Catalog)
行数 (Rows): 4486
列数 (Columns): 103
列名: ['TARGETID', 'COADD_FIBERSTATUS', 'TARGET_RA', 'TARGET_DEC', 'DESINAME', 'PMRA', 'PMDEC', 'REF_EPOCH', 'FA_TARGET', 'FA_TYPE', 'OBJTYPE', 'SUBPRIORITY', 'OBSCONDITIONS', 

In [3]:
analyze_fits("../CLAUDS_udrop_specz/COSMOS_udrop.fits")

--- 正在分析文件: ../CLAUDS_udrop_specz/COSMOS_udrop.fits ---

【1. 文件结构 (HDU List)】
Filename: ../CLAUDS_udrop_specz/COSMOS_udrop.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  Joined        1 BinTableHDU     24   14091R x 6C   [D, D, E, E, E, L]   


--- 扩展层 (Extension) 0 详情 ---
----------------------------------------

--- 扩展层 (Extension) 1 详情 ---
类型: 二进制表格 (Binary Table/Catalog)
行数 (Rows): 14091
列数 (Columns): 6
列名: ['RA', 'DEC', 'PMRA', 'PMDEC', 'REF_EPOCH', 'OVERRIDE']
----------------------------------------



In [4]:
analyze_fits("../CLAUDS_udrop_specz/XMM_udrop_specz.fits")

--- 正在分析文件: ../CLAUDS_udrop_specz/XMM_udrop_specz.fits ---

【1. 文件结构 (HDU List)】
Filename: ../CLAUDS_udrop_specz/XMM_udrop_specz.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  SPECINFO      1 BinTableHDU    286   6386R x 103C   [K, J, D, D, 22A, E, E, E, K, B, 3A, D, J, E, K, K, K, K, K, K, K, K, K, D, D, I, E, I, I, E, E, E, E, E, D, E, D, E, D, D, D, K, 108A, K, 10A, 209A, E, E, L, L, L, 30A, 100A, L, L, L, E, E, 10A, 450A, D, D, K, D, 10D, 4A, K, 6A, 20A, K, D, 9D, 54A, 9D, 120A, D, L, D, D, D, D, D, D, D, D, L, D, D, D, D, D, D, D, D, L, D, D, D, D, D, D, D, D]   


--- 扩展层 (Extension) 0 详情 ---
----------------------------------------

--- 扩展层 (Extension) 1 详情 ---
类型: 二进制表格 (Binary Table/Catalog)
行数 (Rows): 6386
列数 (Columns): 103
列名: ['TARGETID', 'COADD_FIBERSTATUS', 'TARGET_RA', 'TARGET_DEC', 'DESINAME', 'PMRA', 'PMDEC', 'REF_EPOCH', 'FA_TARGET', 'FA_TYPE', 'OBJTYPE', 'SUBPRIORITY', 'OBSCONDITIONS', 'EBV',

In [5]:
analyze_fits("../CLAUDS_udrop_specz/XMM_udrop.fits")

--- 正在分析文件: ../CLAUDS_udrop_specz/XMM_udrop.fits ---

【1. 文件结构 (HDU List)】
Filename: ../CLAUDS_udrop_specz/XMM_udrop.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  XMM_LSS_v10_v210913_withGALEX.fits#1    1 BinTableHDU    460   11664R x 214C   [K, D, D, K, 3A, I, I, 7L, E, E, E, E, E, E, D, E, E, E, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, D, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, K, D, D, D, K, K, K, D, D, D, D, D, D, D, K, D, D, K, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, K, K, D, D, D, D, D, D, D, D, D, D, K, K, K, K, L, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, L, L, L]   


--- 扩展层 (Extension) 0 详情 ---
--